# Requirement 5: Missed-Settlement Cluster Analysis

This notebook runs the approved Global and Local Moran's I workflow using PostGIS-resident baseline reconciliation data. It identifies spatial concentration of the GPS-derived missed indicator; it does not establish cause, verify individual visits, or infer vaccination outcomes.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Run this notebook from the Q1 project or notebooks directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\umary\Desktop\EHA test\Q1_Campaign_Team_Tracking


## Approved analysis design

Primary: 2,382 settlements, excluding 180 ambiguous GPS classifications; `missed_indicator=1` for unvisited and `0` for visited. Weights are binary, row-standardized k-nearest neighbours (`k=8`) in EPSG:32632. Sensitivities use a 9,050 m distance band and inclusion of all 2,562 settlements with ambiguous cases treated as missed. Inference uses 999 permutations, seed 20260730, alpha 0.05, and Benjamini-Hochberg FDR adjustment.

In [2]:
from src.spatial_statistics.moran_analysis import execute_requirement_five

results = execute_requirement_five()
results['global']

C:\ProgramData\anaconda3\envs\eha_gis_assessment\Lib\site-packages\libpysal\weights\distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


,analysis_scenario,weights_specification,observations,missed_settlements,visited_settlements,permutations,seed,global_moran_i,expected_i,z_score,permutation_p_value
0,primary_knn8_excluding_ambiguous,knn_k8_binary_row_standardized,2382,2168,214,999,20260730,0.046612,-0.00042,4.821064,0.001
1,sensitivity_distance_band_excluding_ambiguous,distance_band_9050m_binary_row_standardized,2382,2168,214,999,20260730,0.040196,-0.00042,8.401129,0.001
2,sensitivity_knn8_ambiguous_as_missed,knn_k8_binary_row_standardized,2562,2348,214,999,20260730,0.043926,-0.00039,5.004061,0.001


## Local results and interpretation

Raw permutation results are retained for diagnostic transparency. The primary Local Moran labels use FDR-adjusted significance, so multiple local tests are not interpreted as independent discoveries. A non-significant label is not proof of no operational concern.

In [3]:
results['cluster'].sort_values(['analysis_scenario', 'inference', 'cluster_class'])

,analysis_scenario,weights_specification,inference,cluster_class,settlement_count
11,primary_knn8_excluding_ambiguous,knn_k8_binary_row_standardized,fdr_adjusted,Not significant,2382
0,primary_knn8_excluding_ambiguous,knn_k8_binary_row_standardized,raw_permutation,High-Low outlier,1
1,primary_knn8_excluding_ambiguous,knn_k8_binary_row_standardized,raw_permutation,Low-High outlier,15
2,primary_knn8_excluding_ambiguous,knn_k8_binary_row_standardized,raw_permutation,Low-Low visited cluster,58
3,primary_knn8_excluding_ambiguous,knn_k8_binary_row_standardized,raw_permutation,Not significant,2308
12,sensitivity_distance_band_excluding_ambiguous,distance_band_9050m_binary_row_standardized,fdr_adjusted,Not significant,2382
4,sensitivity_distance_band_excluding_ambiguous,distance_band_9050m_binary_row_standardized,raw_permutation,High-Low outlier,8
5,sensitivity_distance_band_excluding_ambiguous,distance_band_9050m_binary_row_standardized,raw_permutation,Low-High outlier,38
6,sensitivity_distance_band_excluding_ambiguous,distance_band_9050m_binary_row_standardized,raw_permutation,Low-Low visited cluster,118
7,sensitivity_distance_band_excluding_ambiguous,distance_band_9050m_binary_row_standardized,raw_permutation,Not significant,2218


## Validation

The execution validates unique settlement IDs within each scenario, valid source geometry, preserved islands, fixed-seed reproducibility, and reconciliation of local class counts to each analysis population. The four CSV tables are written to `outputs/tables/`.

In [4]:
results['diagnostics']

,weights_specification,observations,minimum_neighbours,median_neighbours,maximum_neighbours,islands,connected_components,median_kth_neighbour_distance_m,maximum_kth_neighbour_distance_m,analysis_scenario
0,knn_k4_binary_row_standardized,2382,4,4.0,4,0,2,1767.580753,10014.407577,primary_excluding_ambiguous
1,knn_k6_binary_row_standardized,2382,6,6.0,6,0,1,2196.915211,10451.842080,primary_excluding_ambiguous
2,knn_k8_binary_row_standardized,2382,8,8.0,8,0,1,2614.577082,12285.272837,primary_excluding_ambiguous
3,knn_k10_binary_row_standardized,2382,10,10.0,10,0,1,3011.706714,12510.526590,primary_excluding_ambiguous
4,distance_band_9050m_binary_row_standardized,2382,1,45.0,108,0,1,NaN,NaN,sensitivity_distance_band_excluding_ambiguous
5,knn_k8_binary_row_standardized,2562,8,8.0,8,0,1,2479.086181,12285.272837,sensitivity_knn8_ambiguous_as_missed
